# 🤖 Ollama LLM Setup sur Kaggle Notebooks

Ce notebook installe automatiquement:
- **Ollama** - Runtime pour LLMs
- **Modèle Qwen 7B** - Modèle LLM haute performance
- **Cloudflare Tunnel** - Accès public (optionnel)

**Temps estimé**: 15-20 minutes

---

## 📋 Étape 1: Installation des dépendances

Installation de Zstandard (pour compression) et mise à jour du système

In [ ]:
import subprocess
import time

print("[1/5] Mise à jour du système...")
subprocess.run("apt-get update", shell=True, capture_output=True)
print("✓ Système mis à jour")

print("\n[2/5] Installation de Zstandard...")
subprocess.run("apt-get install -y zstd", shell=True, capture_output=True)
print("✓ Zstandard installé")

## 🔧 Étape 2: Installation d'Ollama

In [ ]:
print("[3/5] Installation d'Ollama...")
result = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Ollama installé")
else:
    print(f"⚠️ Attention: {result.stderr}")

# Vérifier l'installation
result = subprocess.run("ollama --version", shell=True, capture_output=True, text=True)
print(f"Version: {result.stdout.strip()}")

## 🚀 Étape 3: Démarrage d'Ollama

In [ ]:
print("[4/5] Démarrage du service Ollama...")

# Démarrer Ollama en arrière-plan
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print(f"Ollama en cours de démarrage (PID: {ollama_process.pid})...")
time.sleep(5)

# Vérifier que le service écoute
for i in range(10):
    try:
        import socket
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', 11434))
        sock.close()
        if result == 0:
            print("✓ Ollama en écoute sur le port 11434")
            break
    except:
        time.sleep(1)
else:
    print("✓ Service Ollama démarré")

print(f"\n✓ Ollama démarré avec succès!")

## 🧠 Étape 4: Téléchargement du modèle LLM

Cela va télécharger le modèle Qwen 7B (~5.2 GB)

In [ ]:
print("[5/5] Téléchargement du modèle Qwen 7B...")
print("(Cela peut prendre 10-20 minutes selon votre connexion)\n")

# Télécharger le modèle
result = subprocess.run(
    "ollama pull qwen:7b",
    shell=True,
    capture_output=False,
    text=True
)

if result.returncode == 0:
    print("\n✓ Modèle Qwen 7B téléchargé avec succès!")
else:
    print("\n⚠️ Attention lors du téléchargement")

## ✅ Étape 5: Test du modèle

Envoyons une requête de test au modèle

In [ ]:
print("Test du modèle...\n")
print("─" * 60)

prompt = "Bonjour, qui es-tu et quel est ton rôle?"

result = subprocess.run(
    f'echo "{prompt}" | ollama run qwen:7b',
    shell=True,
    capture_output=True,
    text=True,
    timeout=120
)

print(f"Prompt: {prompt}\n")
print(f"Réponse:\n{result.stdout}")
print("─" * 60)
print("\n✓ Modèle fonctionne correctement!")

## 📚 Utiliser l'API REST d'Ollama

In [ ]:
import requests
import json

print("Test de l'API REST d'Ollama...\n")

url = "http://localhost:11434/api/generate"

data = {
    "model": "qwen:7b",
    "prompt": "Explique-moi brièvement ce qu'est un modèle de langage",
    "stream": False
}

try:
    response = requests.post(url, json=data, timeout=60)
    result = response.json()
    
    print("Réponse de l'API:")
print("─" * 60)
    print(result['response'])
    print("─" * 60)
    
    print(f"\nStatistiques:")
    print(f"  • Temps d'exécution: {result['eval_duration']/1e9:.2f}s")
    print(f"  • Tokens générés: {result['eval_count']}")
except Exception as e:
    print(f"Erreur: {str(e)}")

## 🌐 Optionnel: Configuration du tunnel Cloudflare

Cela rendra Ollama accessible via une URL publique

In [ ]:
print("Installation de Cloudflared...\n")

# Télécharger et installer Cloudflared
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
    shell=True,
    capture_output=True
)

subprocess.run(
    "dpkg -i cloudflared-linux-amd64.deb",
    shell=True,
    capture_output=True
)

subprocess.run("rm cloudflared-linux-amd64.deb", shell=True, capture_output=True)

print("✓ Cloudflared installé")

In [ ]:
print("Création du tunnel Cloudflare...\n")
print("=" * 70)

cloudflared = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:11434",
        "--http-host-header", "localhost:11434"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(8)

print("Sortie du tunnel:\n")
for i in range(50):
    line = cloudflared.stdout.readline()
    if line:
        print(line.rstrip())
        if "trycloudflare.com" in line:
            print("\n✓ URL PUBLIQUE TROUVÉE - voir ci-dessus!")

print("=" * 70)

## 🎯 Résumé et prochaines étapes

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════╗
║        ✓ SETUP COMPLÈTE AVEC SUCCÈS!                         ║
╚══════════════════════════════════════════════════════════════╝

📋 INFORMATIONS DE CONNEXION:

  🖥️  Ollama Local:
     • URL: http://localhost:11434
     • Modèle: Qwen 7B
     • Port: 11434
  
  🌐 Tunnel Cloudflare Public:
     • Voir la sortie ci-dessus pour l'URL publique
     • Reste actif tant que ce notebook s'exécute

📝 COMMANDES UTILES:

  # Lister les modèles
  !ollama list
  
  # Exécuter le modèle en terminal
  !ollama run qwen:7b
  
  # Télécharger un autre modèle
  !ollama pull mistral:7b
  !ollama pull llama2:7b

🔧 MODÈLES ALTERNATIFS:

  - qwen:7b          (Recommandé - équilibre optimal)
  - mistral:7b       (Très rapide - 4.1 GB)
  - llama2:7b        (Populaire - 3.8 GB)
  - neural-chat:7b   (Chat optimisé - 4.7 GB)
  - orca2:13b        (Haute performance - 7.7 GB)

📚 RESSOURCES:

  • Documentation Ollama: https://ollama.com
  • Modèles HuggingFace: https://huggingface.co
  • API Ollama: http://localhost:11434/api

💡 EXEMPLE D'UTILISATION:

  import requests
  url = "http://localhost:11434/api/generate"
  response = requests.post(url, json={
      "model": "qwen:7b",
      "prompt": "Votre question ici",
      "stream": False
  })
  print(response.json()['response'])
"""

print(summary)

## 🔌 Fonction helper pour utiliser facilement Ollama

In [ ]:
import subprocess
import requests
import json

class OllamaHelper:
    """Classe helper pour utiliser Ollama facilement"""
    
    def __init__(self, model="qwen:7b", base_url="http://localhost:11434"):
        self.model = model
        self.base_url = base_url
    
    def generate(self, prompt, temperature=0.7):
        """Générer du texte à partir d'un prompt"""
        url = f"{self.base_url}/api/generate"
        data = {
            "model": self.model,
            "prompt": prompt,
            "temperature": temperature,
            "stream": False
        }
        try:
            response = requests.post(url, json=data, timeout=300)
            return response.json()['response']
        except Exception as e:
            return f"Erreur: {str(e)}"
    
    def chat(self, messages):
        """Chat avec historique"""
        url = f"{self.base_url}/api/chat"
        data = {
            "model": self.model,
            "messages": messages,
            "stream": False
        }
        try:
            response = requests.post(url, json=data, timeout=300)
            return response.json()['message']['content']
        except Exception as e:
            return f"Erreur: {str(e)}"
    
    def list_models(self):
        """Lister les modèles disponibles"""
        url = f"{self.base_url}/api/tags"
        try:
            response = requests.get(url)
            return response.json()['models']
        except Exception as e:
            return f"Erreur: {str(e)}"

# Utilisation
ollama = OllamaHelper(model="qwen:7b")

# Test
print("Génération:")
response = ollama.generate("Donne-moi 3 faits intéressants sur l'espace")
print(response)